# Track 2: Pretrained Audio Model Fine-tuning for SER

Fine-tunes **Wav2Vec2-base** (`facebook/wav2vec2-base`) on EmoDB for
7-class speech emotion recognition.

- Input: raw waveform at 16 kHz (no manual feature extraction)
- Utterance-level model — one prediction per utterance, no segmentation
- Evaluation: LOSO 10-fold, utterance-level accuracy

In [1]:
import os
import copy

import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model

from ser_utils import (
    NUM_CLASSES,
    DEFAULT_SPEAKER_ORDER,
    SplitConfig,
    build_loso_split_configs,
    collect_utterances,
    set_all_seeds,
    evaluate_and_report,
    compute_class_weights,
)

set_all_seeds(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

d:\Env\micromamba\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## Configuration

In [2]:
# ── Paths ──────────────────────────────────────────────────────────
EMODB_DIR       = "../emodb"  # raw wav files
MODEL_DIR       = "models"
RESULTS_DIR     = "results"
SPLIT_MODE      = "loso"
NUM_FOLDS       = 10

# ── Pretrained model ──────────────────────────────────────────────
PRETRAINED_NAME = "facebook/wav2vec2-base"   # ~95 M params, 16 kHz
HIDDEN_SIZE     = 768                        # Wav2Vec2-base hidden dim
SAMPLE_RATE     = 16000
MAX_AUDIO_LEN   = 10 * SAMPLE_RATE           # 10 s ceiling

# ── Training ──────────────────────────────────────────────────────
BATCH_SIZE      = 8
GRAD_ACCUM      = 4                          # effective batch = 32
MAX_EPOCHS      = 20
LR_PRETRAINED   = 1e-5
LR_HEAD         = 1e-3
WEIGHT_DECAY    = 0.01
WARMUP_STEPS    = 100
PATIENCE        = 5
MAX_GRAD_NORM   = 1.0
USE_FP16        = torch.cuda.is_available()

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

## Dataset

In [3]:
class EmoDBWaveformDataset(Dataset):
    """
    Loads raw EmoDB waveforms at 16 kHz.
    Each item: (waveform_tensor, label, utterance_id).
    """

    def __init__(self, entries):
        """
        Args:
            entries: list of (filename, speaker_id, label, file_path)
        """
        self.entries = entries

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        filename, _, label, file_path = self.entries[idx]
        wav, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        # Truncate if too long
        if len(wav) > MAX_AUDIO_LEN:
            wav = wav[:MAX_AUDIO_LEN]
        return torch.from_numpy(wav).float(), label, filename


def waveform_collate_fn(batch):
    """Pad waveforms to max length in batch; create attention mask."""
    waveforms, labels, utt_ids = zip(*batch)
    lengths = [w.size(0) for w in waveforms]
    max_len = max(lengths)

    padded = torch.zeros(len(waveforms), max_len)
    mask   = torch.zeros(len(waveforms), max_len, dtype=torch.long)
    for i, (w, l) in enumerate(zip(waveforms, lengths)):
        padded[i, :l] = w
        mask[i, :l] = 1

    labels = torch.tensor(labels, dtype=torch.long)
    return padded, mask, labels, list(utt_ids)


def build_waveform_dataloaders(fold_config: SplitConfig, emodb_dir: str):
    """Build train / val / test DataLoaders for one LOSO fold."""
    all_utts = collect_utterances(emodb_dir)
    splits = {"train": [], "validation": [], "test": []}
    for entry in all_utts:
        _, speaker_id, _, _ = entry
        s = fold_config.get_split_name(speaker_id)
        if s is not None:
            splits[s].append(entry)

    loaders = {}
    for split_name, entries in splits.items():
        ds = EmoDBWaveformDataset(entries)
        loaders[split_name] = DataLoader(
            ds,
            batch_size=BATCH_SIZE,
            shuffle=(split_name == "train"),
            collate_fn=waveform_collate_fn,
            drop_last=False,
        )
    return loaders["train"], loaders["validation"], loaders["test"]

## Model: Wav2Vec2ForSER

In [4]:
class Wav2Vec2ForSER(nn.Module):
    """
    Wav2Vec2-base + mean pooling + classification head.

    - CNN feature extractor: FROZEN (general speech features)
    - Transformer encoder:   FINE-TUNED with small LR
    - Classification head:   trained with larger LR
    """

    def __init__(self, pretrained_name: str = PRETRAINED_NAME, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(pretrained_name)
        # Freeze CNN feature extractor
        self.wav2vec2.feature_extractor._freeze_parameters()

        self.classifier = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes),
        )

    def forward(self, input_values, attention_mask=None):
        outputs = self.wav2vec2(
            input_values=input_values,
            attention_mask=attention_mask,
        )
        hidden = outputs.last_hidden_state       # (B, T', 768)

        # Build mask at the hidden-state resolution
        if attention_mask is not None:
            # Wav2Vec2 CNN downsamples by ~320x; use extract_features output length
            # to determine the actual hidden-state length
            T_hidden = hidden.size(1)
            # Downsample attention mask to match hidden states
            # Simple approach: take evenly-spaced indices
            T_input = attention_mask.size(1)
            ratio = T_input / T_hidden
            indices = (torch.arange(T_hidden, device=hidden.device) * ratio).long()
            indices = indices.clamp(max=T_input - 1)
            mask = attention_mask[:, indices].unsqueeze(-1).float()  # (B, T', 1)
        else:
            mask = torch.ones(hidden.size(0), hidden.size(1), 1, device=hidden.device)

        # Masked mean pooling
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-8)  # (B, 768)

        return self.classifier(pooled)           # (B, num_classes)


# Quick check
_m = Wav2Vec2ForSER()
total = sum(p.numel() for p in _m.parameters())
trainable = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"Wav2Vec2ForSER: {total:,} total params, {trainable:,} trainable")
del _m

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 64659.76it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Wav2Vec2ForSER: 94,570,375 total params, 90,369,927 trainable


## Training Process

In [5]:
def make_optimizer(model):
    """AdamW with differential learning rates."""
    pretrained_params = list(model.wav2vec2.parameters())
    head_params = list(model.classifier.parameters())
    return optim.AdamW(
        [
            {"params": pretrained_params, "lr": LR_PRETRAINED},
            {"params": head_params,       "lr": LR_HEAD},
        ],
        weight_decay=WEIGHT_DECAY,
    )


def get_scheduler(optimizer, num_training_steps):
    """Linear warmup then cosine decay."""
    from torch.optim.lr_scheduler import LambdaLR
    import math

    def lr_lambda(step):
        if step < WARMUP_STEPS:
            return step / max(1, WARMUP_STEPS)
        progress = (step - WARMUP_STEPS) / max(1, num_training_steps - WARMUP_STEPS)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    return LambdaLR(optimizer, lr_lambda)


def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, global_step):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad()

    for batch_idx, (wav, mask, labels, _) in enumerate(loader):
        wav, mask, labels = wav.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)

        with torch.amp.autocast("cuda", enabled=USE_FP16):
            logits = model(wav, attention_mask=mask)
            loss = criterion(logits, labels) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            global_step += 1

        total_loss += loss.item() * GRAD_ACCUM * wav.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += wav.size(0)

    return total_loss / total, correct / total, global_step


@torch.no_grad()
def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_utt_ids = [], [], []

    for wav, mask, labels, utt_ids in loader:
        wav, mask, labels = wav.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=USE_FP16):
            logits = model(wav, attention_mask=mask)
            loss = criterion(logits, labels)
        preds = logits.argmax(1)
        total_loss += loss.item() * wav.size(0)
        correct += (preds == labels).sum().item()
        total += wav.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_utt_ids.extend(utt_ids)

    return (
        total_loss / total,
        correct / total,
        np.array(all_preds),
        np.array(all_labels),
        np.array(all_utt_ids),
    )


def train_one_fold(fold_config: SplitConfig, fold_idx: int):
    """Train on one LOSO fold; return (utt_preds, utt_true, utt_ids)."""
    print(f"\n{'─' * 50}")
    print(f"  Fold {fold_idx:02d}  test={fold_config.test_speakers[0]}  val={fold_config.validation_speakers[0]}")
    print(f"{'─' * 50}")

    train_loader, val_loader, test_loader = build_waveform_dataloaders(fold_config, EMODB_DIR)

    # Compute class weights from training labels
    train_labels = np.array([entry[2] for entry in train_loader.dataset.entries])
    weights = compute_class_weights(train_labels).to(DEVICE)

    model = Wav2Vec2ForSER().to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = make_optimizer(model)

    num_training_steps = (len(train_loader) // GRAD_ACCUM + 1) * MAX_EPOCHS
    scheduler = get_scheduler(optimizer, num_training_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)

    best_val_loss = float("inf")
    best_state = None
    wait = 0
    global_step = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_acc, global_step = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, scaler, global_step
        )
        val_loss, val_acc, _, _, _ = evaluate_model(model, val_loader, criterion)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        print(
            f"  Epoch {epoch:2d} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} | "
            f"wait={wait}"
        )

        if wait >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}")
            break

    # Save best model
    ckpt_path = os.path.join(MODEL_DIR, f"wav2vec2_fold{fold_idx:02d}.pth")
    torch.save(best_state, ckpt_path)
    print(f"  Saved → {ckpt_path}")

    # Evaluate on test set
    model.load_state_dict(best_state)
    _, test_acc, preds, labels, utt_ids = evaluate_model(model, test_loader, criterion)
    print(f"  Test utterance acc: {test_acc:.3f}")

    return preds, labels, utt_ids

## LOSO Training Loop

In [6]:
folds = build_loso_split_configs(DEFAULT_SPEAKER_ORDER)

all_preds  = []
all_labels = []
all_utt_ids = []

for fold_idx, fold_cfg in enumerate(folds, start=1):
    preds, labels, utt_ids = train_one_fold(fold_cfg, fold_idx)
    all_preds.append(preds)
    all_labels.append(labels)
    all_utt_ids.append(utt_ids)

all_preds   = np.concatenate(all_preds)
all_labels  = np.concatenate(all_labels)
all_utt_ids = np.concatenate(all_utt_ids)

print(f"\nTotal test utterances: {len(all_preds)}")


──────────────────────────────────────────────────
  Fold 01  test=03  val=08
──────────────────────────────────────────────────


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 60958.68it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch  1 | train_loss=1.9462 train_acc=0.136 | val_loss=1.9561 val_acc=0.052 | wait=0
  Epoch  2 | train_loss=1.9034 train_acc=0.297 | val_loss=1.8622 val_acc=0.379 | wait=0
  Epoch  3 | train_loss=1.8186 train_acc=0.409 | val_loss=1.7090 val_acc=0.362 | wait=0
  Epoch  4 | train_loss=1.6401 train_acc=0.512 | val_loss=1.4524 val_acc=0.414 | wait=0
  Epoch  5 | train_loss=1.3146 train_acc=0.610 | val_loss=1.1350 val_acc=0.552 | wait=0
  Epoch  6 | train_loss=0.9700 train_acc=0.696 | val_loss=0.8801 val_acc=0.672 | wait=0
  Epoch  7 | train_loss=0.6574 train_acc=0.815 | val_loss=0.7593 val_acc=0.741 | wait=0
  Epoch  8 | train_loss=0.4390 train_acc=0.860 | val_loss=0.7289 val_acc=0.690 | wait=0
  Epoch  9 | train_loss=0.3247 train_acc=0.895 | val_loss=0.5179 val_acc=0.741 | wait=0
  Epoch 10 | train_loss=0.2267 train_acc=0.923 | val_loss=0.3968 val_acc=0.828 | wait=0
  Epoch 11 | train_loss=0.1332 train_acc=0.953 | val_loss=0.3887 val_acc=0.845 | wait=0
  Epoch 12 | train_loss=0.1028 t

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 46315.58it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch  1 | train_loss=1.9403 train_acc=0.141 | val_loss=1.9206 val_acc=0.093 | wait=0
  Epoch  2 | train_loss=1.9033 train_acc=0.302 | val_loss=1.8490 val_acc=0.419 | wait=0
  Epoch  3 | train_loss=1.8069 train_acc=0.475 | val_loss=1.7183 val_acc=0.419 | wait=0
  Epoch  4 | train_loss=1.6358 train_acc=0.525 | val_loss=1.4554 val_acc=0.581 | wait=0
  Epoch  5 | train_loss=1.3361 train_acc=0.657 | val_loss=1.2004 val_acc=0.581 | wait=0
  Epoch  6 | train_loss=0.9760 train_acc=0.707 | val_loss=0.7960 val_acc=0.744 | wait=0
  Epoch  7 | train_loss=0.6672 train_acc=0.800 | val_loss=0.6696 val_acc=0.791 | wait=0
  Epoch  8 | train_loss=0.4512 train_acc=0.846 | val_loss=0.4670 val_acc=0.791 | wait=0
  Epoch  9 | train_loss=0.3270 train_acc=0.873 | val_loss=0.3803 val_acc=0.837 | wait=0
  Epoch 10 | train_loss=0.2098 train_acc=0.935 | val_loss=0.4593 val_acc=0.791 | wait=1
  Epoch 11 | train_loss=0.1626 train_acc=0.938 | val_loss=0.2604 val_acc=0.884 | wait=0
  Epoch 12 | train_loss=0.0828 t

...

  Epoch  1 | train_loss=1.9458 train_acc=0.163 | val_loss=1.9310 val_acc=0.161 | wait=0
  Epoch  2 | train_loss=1.9155 train_acc=0.224 | val_loss=1.8874 val_acc=0.339 | wait=0
  Epoch  3 | train_loss=1.8520 train_acc=0.434 | val_loss=1.7858 val_acc=0.518 | wait=0
  Epoch  4 | train_loss=1.7283 train_acc=0.444 | val_loss=1.5551 val_acc=0.571 | wait=0
  Epoch  5 | train_loss=1.4682 train_acc=0.588 | val_loss=1.2005 val_acc=0.679 | wait=0
  Epoch  6 | train_loss=1.1398 train_acc=0.617 | val_loss=0.9417 val_acc=0.768 | wait=0
  Epoch  7 | train_loss=0.8742 train_acc=0.712 | val_loss=0.6458 val_acc=0.804 | wait=0
  Epoch  8 | train_loss=0.6602 train_acc=0.766 | val_loss=0.6138 val_acc=0.875 | wait=0
  Epoch  9 | train_loss=0.4209 train_acc=0.873 | val_loss=0.7254 val_acc=0.857 | wait=1
  Epoch 10 | train_loss=0.3056 train_acc=0.905 | val_loss=0.7189 val_acc=0.893 | wait=2
  Epoch 11 | train_loss=0.2253 train_acc=0.920 | val_loss=0.8972 val_acc=0.804 | wait=3
  Epoch 12 | train_loss=0.2342 t

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 43887.83it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch  1 | train_loss=1.9456 train_acc=0.260 | val_loss=1.9430 val_acc=0.197 | wait=0
  Epoch  2 | train_loss=1.9169 train_acc=0.279 | val_loss=1.8896 val_acc=0.380 | wait=0
  Epoch  3 | train_loss=1.8559 train_acc=0.441 | val_loss=1.7667 val_acc=0.521 | wait=0
  Epoch  4 | train_loss=1.7255 train_acc=0.556 | val_loss=1.5583 val_acc=0.549 | wait=0
  Epoch  5 | train_loss=1.4867 train_acc=0.620 | val_loss=1.2299 val_acc=0.704 | wait=0
  Epoch  6 | train_loss=1.1331 train_acc=0.696 | val_loss=0.8554 val_acc=0.761 | wait=0
  Epoch  7 | train_loss=0.7719 train_acc=0.760 | val_loss=0.5986 val_acc=0.761 | wait=0
  Epoch  8 | train_loss=0.4991 train_acc=0.863 | val_loss=0.5606 val_acc=0.761 | wait=0
  Epoch  9 | train_loss=0.3481 train_acc=0.882 | val_loss=0.4259 val_acc=0.845 | wait=0
  Epoch 10 | train_loss=0.2836 train_acc=0.897 | val_loss=0.8356 val_acc=0.676 | wait=1
  Epoch 11 | train_loss=0.2022 train_acc=0.922 | val_loss=0.5017 val_acc=0.817 | wait=2
  Epoch 12 | train_loss=0.1817 t

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 41601.94it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch  1 | train_loss=1.9438 train_acc=0.205 | val_loss=1.9175 val_acc=0.327 | wait=0
  Epoch  2 | train_loss=1.9122 train_acc=0.386 | val_loss=1.8831 val_acc=0.510 | wait=0
  Epoch  3 | train_loss=1.8458 train_acc=0.427 | val_loss=1.7645 val_acc=0.551 | wait=0
  Epoch  4 | train_loss=1.6872 train_acc=0.492 | val_loss=1.5354 val_acc=0.653 | wait=0
  Epoch  5 | train_loss=1.4410 train_acc=0.612 | val_loss=1.2183 val_acc=0.592 | wait=0
  Epoch  6 | train_loss=1.1278 train_acc=0.651 | val_loss=0.8875 val_acc=0.776 | wait=0
  Epoch  7 | train_loss=0.7743 train_acc=0.807 | val_loss=0.6283 val_acc=0.857 | wait=0
  Epoch  8 | train_loss=0.5425 train_acc=0.817 | val_loss=0.4852 val_acc=0.816 | wait=0
  Epoch  9 | train_loss=0.3614 train_acc=0.880 | val_loss=0.5713 val_acc=0.857 | wait=1
  Epoch 10 | train_loss=0.3098 train_acc=0.880 | val_loss=0.3533 val_acc=0.878 | wait=0
  Epoch 11 | train_loss=0.2382 train_acc=0.916 | val_loss=0.3612 val_acc=0.939 | wait=1
  Epoch 12 | train_loss=0.1738 t

## Evaluation — Utterance-Level

In [7]:
# Wav2Vec2 is already utterance-level, no majority voting needed.
acc, report, cm = evaluate_and_report(
    all_labels, all_preds,
    model_name="Wav2Vec2-base",
    results_dir=RESULTS_DIR,
    split_mode=SPLIT_MODE,
)


  Wav2Vec2-base  —  LOSO
  Utterance-level accuracy: 84.49%
              precision    recall  f1-score   support

       Anger       0.90      0.88      0.89       127
     Boredom       0.86      0.84      0.85        81
        Fear       0.88      0.71      0.78        69
   Happiness       0.62      0.72      0.67        71
     Sadness       0.97      0.95      0.96        62
     Disgust       0.89      0.91      0.90        46
     Neutral       0.83      0.90      0.86        79

    accuracy                           0.84       535
   macro avg       0.85      0.84      0.85       535
weighted avg       0.85      0.84      0.85       535

  Confusion matrix saved → results\cm_Wav2Vec2-base_loso.png


## Ablations: (i) attnstat pooling (ii) Other pretrained models

In [8]:
import csv
import gc
import json
import math
import time
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import AutoModel, WhisperFeatureExtractor, WhisperForAudioClassification

from ser_utils import EMOTION_ENG_MAP


AF_RESULTS_DIR = RESULTS_DIR
AF_MODEL_DIR = MODEL_DIR
AF_SPLIT_MODE = SPLIT_MODE
AF_LABEL_NAMES = [EMOTION_ENG_MAP[i] for i in range(NUM_CLASSES)]
AF_PROB_COLS = [f"prob_{i}" for i in range(NUM_CLASSES)]


def af_safe_name(name: str) -> str:
    return (
        name.replace("/", "_")
        .replace(" ", "_")
        .replace("-", "-")
        .replace(".", "p")
    )


@dataclass
class AFExperimentConfig:
    short_name: str
    model_id: str
    family: str = "ssl"  # ssl or whisper
    pooling: str = "attentive_stats"
    batch_size: int = 4
    grad_accum: int = 8
    max_epochs: int = 18
    patience: int = 5
    pretrained_lr: float = 1e-5
    head_lr: float = 5e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.08
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.05
    max_audio_len: int = 10 * SAMPLE_RATE
    normalize_waveform: bool = False
    freeze_feature_extractor: bool = True
    freeze_encoder_layers: int = 0
    gradient_checkpointing: bool = False
    use_fp16: bool = torch.cuda.is_available()
    num_workers: int = 0
    smoke_epochs: int = 1


AF_EXPERIMENTS: List[AFExperimentConfig] = [
    AFExperimentConfig(
        short_name="Wav2Vec2-base-attnstat",
        model_id="facebook/wav2vec2-base",
        family="ssl",
        pooling="attentive_stats",
        batch_size=6,
        grad_accum=6,
        max_epochs=20,
        pretrained_lr=1e-5,
        head_lr=7e-4,
    ),
    AFExperimentConfig(
        short_name="WavLM-base-attnstat",
        model_id="microsoft/wavlm-base",
        family="ssl",
        pooling="attentive_stats",
        batch_size=6,
        grad_accum=6,
        max_epochs=20,
        pretrained_lr=1e-5,
        head_lr=7e-4,
    ),
    AFExperimentConfig(
        short_name="Whisper-base",
        model_id="openai/whisper-base",
        family="whisper",
        pooling="hf_classifier",
        batch_size=2,
        grad_accum=12,
        max_epochs=16,
        pretrained_lr=5e-6,
        head_lr=5e-4,
        gradient_checkpointing=True,
    ),
    AFExperimentConfig(
        short_name="HuBERT-base-attnstat",
        model_id="facebook/hubert-base-ls960",
        family="ssl",
        pooling="attentive_stats",
        batch_size=6,
        grad_accum=6,
        max_epochs=18,
        pretrained_lr=1e-5,
        head_lr=7e-4,
    ),
]


def af_build_entries_by_split(fold_config: SplitConfig, emodb_dir: str):
    all_utts = collect_utterances(emodb_dir)
    splits = {"train": [], "validation": [], "test": []}
    for entry in all_utts:
        _, speaker_id, _, _ = entry
        split_name = fold_config.get_split_name(speaker_id)
        if split_name is not None:
            splits[split_name].append(entry)
    return splits


class AFWaveformDataset(Dataset):
    def __init__(self, entries, max_audio_len: int, normalize_waveform: bool = False):
        self.entries = list(entries)
        self.max_audio_len = max_audio_len
        self.normalize_waveform = normalize_waveform

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        filename, _, label, file_path = self.entries[idx]
        wav, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        if len(wav) > self.max_audio_len:
            wav = wav[: self.max_audio_len]
        if self.normalize_waveform:
            wav = wav.astype(np.float32)
            wav = wav - wav.mean()
            wav = wav / (wav.std() + 1e-6)
        return torch.from_numpy(wav).float(), int(label), filename


def af_ssl_collate(batch):
    waveforms, labels, utt_ids = zip(*batch)
    lengths = [w.size(0) for w in waveforms]
    max_len = max(lengths)
    padded = torch.zeros(len(waveforms), max_len)
    mask = torch.zeros(len(waveforms), max_len, dtype=torch.long)
    for i, (wav, length) in enumerate(zip(waveforms, lengths)):
        padded[i, :length] = wav
        mask[i, :length] = 1
    labels = torch.tensor(labels, dtype=torch.long)
    return {"input_values": padded, "attention_mask": mask, "labels": labels, "utt_ids": list(utt_ids)}


class AFWhisperCollator:
    def __init__(self, feature_extractor: WhisperFeatureExtractor):
        self.feature_extractor = feature_extractor
        self.max_samples = getattr(feature_extractor, "n_samples", 30 * SAMPLE_RATE)
        self.max_frames = getattr(feature_extractor, "nb_max_frames", 3000)

    def __call__(self, batch):
        waveforms, labels, utt_ids = zip(*batch)
        arrays = [w.numpy() for w in waveforms]
        features = self.feature_extractor(
            arrays,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_samples,
        )
        input_features = features.input_features
        current_frames = input_features.size(-1)
        if current_frames < self.max_frames:
            input_features = torch.nn.functional.pad(
                input_features,
                (0, self.max_frames - current_frames),
            )
        elif current_frames > self.max_frames:
            input_features = input_features[..., : self.max_frames]
        labels = torch.tensor(labels, dtype=torch.long)
        return {"input_features": input_features, "labels": labels, "utt_ids": list(utt_ids)}


def af_make_loaders(cfg: AFExperimentConfig, fold_config: SplitConfig):
    splits = af_build_entries_by_split(fold_config, EMODB_DIR)
    datasets = {
        split: AFWaveformDataset(
            entries,
            max_audio_len=cfg.max_audio_len,
            normalize_waveform=cfg.normalize_waveform,
        )
        for split, entries in splits.items()
    }
    if cfg.family == "whisper":
        feature_extractor = WhisperFeatureExtractor.from_pretrained(cfg.model_id)
        collate_fn = AFWhisperCollator(feature_extractor)
    else:
        collate_fn = af_ssl_collate

    loaders = {}
    for split_name in ("train", "validation", "test"):
        loaders[split_name] = DataLoader(
            datasets[split_name],
            batch_size=cfg.batch_size,
            shuffle=(split_name == "train"),
            collate_fn=collate_fn,
            drop_last=False,
            num_workers=cfg.num_workers,
            pin_memory=torch.cuda.is_available(),
        )
    return loaders["train"], loaders["validation"], loaders["test"]


def af_downsample_mask(attention_mask: torch.Tensor, target_len: int) -> torch.Tensor:
    if attention_mask is None:
        return None
    input_len = attention_mask.size(1)
    if input_len == target_len:
        return attention_mask.float()
    idx = torch.linspace(0, input_len - 1, steps=target_len, device=attention_mask.device)
    idx = idx.round().long().clamp(min=0, max=input_len - 1)
    return attention_mask[:, idx].float()


class AFAttentiveStatsPooling(nn.Module):
    def __init__(self, hidden_size: int, bottleneck: int = 128):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_size, bottleneck),
            nn.Tanh(),
            nn.Linear(bottleneck, 1),
        )

    def forward(self, hidden: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        scores = self.attn(hidden).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(mask <= 0, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        mean = torch.sum(hidden * weights, dim=1)
        var = torch.sum(((hidden - mean.unsqueeze(1)) ** 2) * weights, dim=1)
        std = torch.sqrt(var.clamp(min=1e-6))
        return torch.cat([mean, std], dim=-1)


class AFSSLForSER(nn.Module):
    def __init__(self, cfg: AFExperimentConfig):
        super().__init__()
        self.cfg = cfg
        self.encoder = AutoModel.from_pretrained(cfg.model_id)
        hidden_size = int(self.encoder.config.hidden_size)

        if cfg.gradient_checkpointing and hasattr(self.encoder, "gradient_checkpointing_enable"):
            self.encoder.gradient_checkpointing_enable()

        if cfg.freeze_feature_extractor:
            feature_extractor = getattr(self.encoder, "feature_extractor", None)
            if feature_extractor is not None and hasattr(feature_extractor, "_freeze_parameters"):
                feature_extractor._freeze_parameters()

        if cfg.freeze_encoder_layers > 0:
            self.freeze_lower_encoder_layers(cfg.freeze_encoder_layers)

        if cfg.pooling == "mean":
            self.pool = None
            pooled_dim = hidden_size
        elif cfg.pooling == "attentive_stats":
            self.pool = AFAttentiveStatsPooling(hidden_size)
            pooled_dim = hidden_size * 2
        else:
            raise ValueError(f"Unsupported SSL pooling: {cfg.pooling}")

        self.classifier = nn.Sequential(
            nn.LayerNorm(pooled_dim),
            nn.Dropout(0.2),
            nn.Linear(pooled_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, NUM_CLASSES),
        )

    def freeze_lower_encoder_layers(self, n_layers: int):
        encoder = getattr(self.encoder, "encoder", None)
        layers = getattr(encoder, "layers", None)
        if layers is None:
            print("  [WARN] Could not locate encoder layers for partial freezing.")
            return
        for layer in layers[:n_layers]:
            for param in layer.parameters():
                param.requires_grad = False
        print(f"  Frozen lower encoder layers: {min(n_layers, len(layers))}/{len(layers)}")

    def forward(self, input_values, attention_mask=None, return_embedding: bool = False):
        outputs = self.encoder(input_values=input_values, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        hidden_mask = af_downsample_mask(attention_mask, hidden.size(1)) if attention_mask is not None else None
        if self.pool is None:
            if hidden_mask is None:
                pooled = hidden.mean(dim=1)
            else:
                denom = hidden_mask.sum(dim=1, keepdim=True).clamp(min=1e-6)
                pooled = (hidden * hidden_mask.unsqueeze(-1)).sum(dim=1) / denom
        else:
            pooled = self.pool(hidden, hidden_mask)
        logits = self.classifier(pooled)
        if return_embedding:
            return logits, pooled
        return logits


def af_build_model(cfg: AFExperimentConfig):
    if cfg.family == "ssl":
        return AFSSLForSER(cfg)
    if cfg.family == "whisper":
        model = WhisperForAudioClassification.from_pretrained(
            cfg.model_id,
            num_labels=NUM_CLASSES,
            label2id={name: i for i, name in enumerate(AF_LABEL_NAMES)},
            id2label={i: name for i, name in enumerate(AF_LABEL_NAMES)},
            ignore_mismatched_sizes=True,
        )
        if cfg.gradient_checkpointing and hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()
        if cfg.freeze_encoder_layers > 0:
            layers = getattr(getattr(model, "model", None), "encoder", None)
            layers = getattr(layers, "layers", None)
            if layers is not None:
                for layer in layers[: cfg.freeze_encoder_layers]:
                    for param in layer.parameters():
                        param.requires_grad = False
                print(f"  Frozen Whisper lower encoder layers: {cfg.freeze_encoder_layers}/{len(layers)}")
        return model
    raise ValueError(f"Unknown family: {cfg.family}")


def af_make_optimizer(model: nn.Module, cfg: AFExperimentConfig):
    head_keywords = ("classifier", "projector", "pool")
    head_params, backbone_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(key in name for key in head_keywords):
            head_params.append(param)
        else:
            backbone_params.append(param)
    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": cfg.pretrained_lr})
    if head_params:
        groups.append({"params": head_params, "lr": cfg.head_lr})
    return optim.AdamW(groups, weight_decay=cfg.weight_decay)


def af_make_scheduler(optimizer, num_training_steps: int, warmup_ratio: float):
    from torch.optim.lr_scheduler import LambdaLR

    warmup_steps = max(1, int(num_training_steps * warmup_ratio))

    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / max(1, num_training_steps - warmup_steps)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    return LambdaLR(optimizer, lr_lambda)


def af_forward_logits(model, batch, family: str, return_embedding: bool = False):
    if family == "ssl":
        input_values = batch["input_values"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        return model(input_values, attention_mask=attention_mask, return_embedding=return_embedding)
    input_features = batch["input_features"].to(DEVICE)
    outputs = model(input_features=input_features, output_hidden_states=return_embedding)
    if return_embedding and outputs.hidden_states is not None:
        embedding = outputs.hidden_states[-1].mean(dim=1)
        return outputs.logits, embedding
    return outputs.logits


def af_train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, cfg, global_step: int):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)

    for batch_idx, batch in enumerate(loader):
        labels = batch["labels"].to(DEVICE)
        with torch.amp.autocast("cuda", enabled=cfg.use_fp16):
            logits = af_forward_logits(model, batch, cfg.family)
            loss = criterion(logits, labels) / cfg.grad_accum

        scaler.scale(loss).backward()

        should_step = (batch_idx + 1) % cfg.grad_accum == 0 or (batch_idx + 1) == len(loader)
        if should_step:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

        total_loss += loss.item() * cfg.grad_accum * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / max(1, total), correct / max(1, total), global_step


@torch.no_grad()
def af_evaluate(model, loader, criterion, cfg, collect_embeddings: bool = False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    preds, labels_all, utt_ids = [], [], []
    probs_all, embeddings_all = [], []

    for batch in loader:
        labels = batch["labels"].to(DEVICE)
        with torch.amp.autocast("cuda", enabled=cfg.use_fp16):
            if collect_embeddings:
                logits, embeddings = af_forward_logits(model, batch, cfg.family, return_embedding=True)
            else:
                logits = af_forward_logits(model, batch, cfg.family)
                embeddings = None
            loss = criterion(logits, labels)
        probs = torch.softmax(logits.float(), dim=1)
        batch_preds = probs.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)
        correct += (batch_preds == labels).sum().item()
        total += labels.size(0)
        preds.extend(batch_preds.cpu().numpy().tolist())
        labels_all.extend(labels.cpu().numpy().tolist())
        utt_ids.extend(batch["utt_ids"])
        probs_all.extend(probs.cpu().numpy().tolist())
        if collect_embeddings and embeddings is not None:
            embeddings_all.extend(embeddings.float().cpu().numpy().tolist())

    y_true = np.array(labels_all)
    y_pred = np.array(preds)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0) if len(y_true) else 0.0
    return {
        "loss": total_loss / max(1, total),
        "accuracy": correct / max(1, total),
        "macro_f1": macro_f1,
        "preds": y_pred,
        "labels": y_true,
        "utt_ids": np.array(utt_ids),
        "probs": np.array(probs_all),
        "embeddings": np.array(embeddings_all) if embeddings_all else None,
    }


def af_write_predictions_csv(path: str, rows: Sequence[Dict]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fieldnames = ["model", "fold", "utterance_id", "y_true", "y_pred", "correct"] + AF_PROB_COLS
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def af_save_report(model_name: str, y_true: np.ndarray, y_pred: np.ndarray, split_mode: str = "loso"):
    os.makedirs(AF_RESULTS_DIR, exist_ok=True)
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    report = classification_report(y_true, y_pred, target_names=AF_LABEL_NAMES, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))

    report_path = os.path.join(AF_RESULTS_DIR, f"report_{model_name}_{split_mode}.txt")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(f"Model: {model_name}\n")
        f.write(f"Split: {split_mode}\n")
        f.write(f"Utterance-level accuracy: {acc * 100:.2f}%\n")
        f.write(f"Macro-F1: {macro_f1:.4f}\n\n")
        f.write(report)

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=AF_LABEL_NAMES,
        yticklabels=AF_LABEL_NAMES,
        ax=ax,
    )
    ax.set_title(f"{model_name} - Confusion Matrix ({split_mode.upper()})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    fig.tight_layout()
    cm_path = os.path.join(AF_RESULTS_DIR, f"cm_{model_name}_{split_mode}.png")
    fig.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return {"model": model_name, "accuracy": acc, "macro_f1": macro_f1, "report_path": report_path, "cm_path": cm_path}


def af_train_one_fold(cfg: AFExperimentConfig, fold_config: SplitConfig, fold_idx: int, max_epochs_override: Optional[int] = None):
    print(f"\n{'=' * 70}")
    print(f"  {cfg.short_name} | Fold {fold_idx:02d} | test={fold_config.test_speakers[0]} val={fold_config.validation_speakers[0]}")
    print(f"{'=' * 70}")

    train_loader, val_loader, test_loader = af_make_loaders(cfg, fold_config)
    train_labels = np.array([entry[2] for entry in train_loader.dataset.entries])
    weights = compute_class_weights(train_labels).to(DEVICE)

    model = af_build_model(cfg).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=cfg.label_smoothing)
    optimizer = af_make_optimizer(model, cfg)
    epochs = max_epochs_override if max_epochs_override is not None else cfg.max_epochs
    steps_per_epoch = max(1, math.ceil(len(train_loader) / cfg.grad_accum))
    scheduler = af_make_scheduler(optimizer, steps_per_epoch * epochs, cfg.warmup_ratio)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_fp16)

    best_score = -float("inf")
    best_state = None
    best_epoch = 0
    wait = 0
    global_step = 0
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc, global_step = af_train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, scaler, cfg, global_step
        )
        val_metrics = af_evaluate(model, val_loader, criterion, cfg)
        selection_score = val_metrics["macro_f1"] - 0.05 * val_metrics["loss"]
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_metrics["loss"],
                "val_acc": val_metrics["accuracy"],
                "val_macro_f1": val_metrics["macro_f1"],
            }
        )

        improved = selection_score > best_score
        if improved:
            best_score = selection_score
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            wait = 0
        else:
            wait += 1

        print(
            f"  Epoch {epoch:02d} | train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
            f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.3f} "
            f"val_macro_f1={val_metrics['macro_f1']:.3f} | wait={wait}"
        )
        if wait >= cfg.patience:
            print(f"  Early stopping at epoch {epoch}")
            break

    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())

    safe_name = af_safe_name(cfg.short_name)
    ckpt_path = os.path.join(AF_MODEL_DIR, f"{safe_name}_fold{fold_idx:02d}.pth")
    torch.save(best_state, ckpt_path)
    model.load_state_dict(best_state)
    test_metrics = af_evaluate(model, test_loader, criterion, cfg, collect_embeddings=True)
    print(
        f"  Best epoch: {best_epoch} | Test acc={test_metrics['accuracy']:.3f} "
        f"macro_f1={test_metrics['macro_f1']:.3f}"
    )

    fold_rows = []
    for uid, y, pred, probs in zip(
        test_metrics["utt_ids"], test_metrics["labels"], test_metrics["preds"], test_metrics["probs"]
    ):
        row = {
            "model": cfg.short_name,
            "fold": fold_idx,
            "utterance_id": uid,
            "y_true": int(y),
            "y_pred": int(pred),
            "correct": int(y == pred),
        }
        row.update({f"prob_{i}": float(probs[i]) for i in range(NUM_CLASSES)})
        fold_rows.append(row)

    fold_meta = {
        "model": cfg.short_name,
        "model_id": cfg.model_id,
        "fold": fold_idx,
        "test_speaker": fold_config.test_speakers[0],
        "validation_speaker": fold_config.validation_speakers[0],
        "best_epoch": best_epoch,
        "best_selection_score": best_score,
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "ckpt_path": ckpt_path,
        "history": history,
    }

    if test_metrics["embeddings"] is not None and len(test_metrics["embeddings"]):
        emb_path = os.path.join(AF_RESULTS_DIR, f"{safe_name}_fold{fold_idx:02d}_embeddings.npz")
        np.savez_compressed(
            emb_path,
            embeddings=test_metrics["embeddings"],
            labels=test_metrics["labels"],
            utt_ids=test_metrics["utt_ids"],
        )
        fold_meta["embeddings_path"] = emb_path

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return fold_meta, fold_rows


def af_run_experiment(cfg: AFExperimentConfig, fold_indices: Optional[Sequence[int]] = None, smoke: bool = False):
    set_all_seeds(42)
    os.makedirs(AF_RESULTS_DIR, exist_ok=True)
    os.makedirs(AF_MODEL_DIR, exist_ok=True)
    folds = build_loso_split_configs(DEFAULT_SPEAKER_ORDER)
    if fold_indices is None:
        fold_indices = range(1, len(folds) + 1)

    all_rows, fold_meta = [], []
    start = time.time()
    for fold_idx in fold_indices:
        max_epochs = cfg.smoke_epochs if smoke else None
        meta, rows = af_train_one_fold(cfg, folds[fold_idx - 1], fold_idx, max_epochs_override=max_epochs)
        fold_meta.append(meta)
        all_rows.extend(rows)

    y_true = np.array([r["y_true"] for r in all_rows])
    y_pred = np.array([r["y_pred"] for r in all_rows])
    safe_name = af_safe_name(cfg.short_name + ("-smoke" if smoke else ""))
    pred_path = os.path.join(AF_RESULTS_DIR, f"{safe_name}_{AF_SPLIT_MODE}_predictions.csv")
    af_write_predictions_csv(pred_path, all_rows)
    report_meta = af_save_report(safe_name, y_true, y_pred, AF_SPLIT_MODE)
    summary = {
        **asdict(cfg),
        **report_meta,
        "prediction_path": pred_path,
        "num_folds": len(list(fold_indices)),
        "num_utterances": len(all_rows),
        "elapsed_minutes": (time.time() - start) / 60,
        "folds": fold_meta,
    }
    meta_path = os.path.join(AF_RESULTS_DIR, f"{safe_name}_{AF_SPLIT_MODE}_meta.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    summary["meta_path"] = meta_path
    print(f"\nSaved predictions: {pred_path}")
    print(f"Saved metadata: {meta_path}")
    return summary


def af_write_summary_csv(summaries: Sequence[Dict], path: str = os.path.join(AF_RESULTS_DIR, "pretrained_loso_summary.csv")):
    fieldnames = [
        "short_name",
        "model_id",
        "accuracy",
        "macro_f1",
        "num_folds",
        "num_utterances",
        "elapsed_minutes",
        "prediction_path",
        "report_path",
        "cm_path",
    ]
    existing = []
    if os.path.exists(path):
        with open(path, newline="", encoding="utf-8") as f:
            existing = list(csv.DictReader(f))
    by_name = {row["short_name"]: row for row in existing if row.get("short_name")}
    for s in summaries:
        by_name[s["short_name"]] = {k: s.get(k, "") for k in fieldnames}
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(by_name.values())
    print(f"Summary CSV saved: {path}")
    return path


def af_load_prediction_csv(path: str):
    rows = []
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            rows.append(row)
    return rows


def af_ensemble_from_prediction_files(prediction_files: Sequence[str], ensemble_name: str, weights: Optional[Sequence[float]] = None):
    if weights is None:
        weights = np.ones(len(prediction_files), dtype=float)
    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum()

    keyed = {}
    for model_weight, path in zip(weights, prediction_files):
        for row in af_load_prediction_csv(path):
            uid = row["utterance_id"]
            probs = np.array([float(row[f"prob_{i}"]) for i in range(NUM_CLASSES)])
            if uid not in keyed:
                keyed[uid] = {"y_true": int(row["y_true"]), "probs": np.zeros(NUM_CLASSES)}
            keyed[uid]["probs"] += model_weight * probs

    y_true, y_pred, rows = [], [], []
    for uid in sorted(keyed):
        probs = keyed[uid]["probs"]
        pred = int(np.argmax(probs))
        true = int(keyed[uid]["y_true"])
        y_true.append(true)
        y_pred.append(pred)
        out = {
            "model": ensemble_name,
            "fold": "",
            "utterance_id": uid,
            "y_true": true,
            "y_pred": pred,
            "correct": int(true == pred),
        }
        out.update({f"prob_{i}": float(probs[i]) for i in range(NUM_CLASSES)})
        rows.append(out)

    safe_name = af_safe_name(ensemble_name)
    pred_path = os.path.join(AF_RESULTS_DIR, f"{safe_name}_{AF_SPLIT_MODE}_predictions.csv")
    af_write_predictions_csv(pred_path, rows)
    report_meta = af_save_report(safe_name, np.array(y_true), np.array(y_pred), AF_SPLIT_MODE)
    return {"short_name": ensemble_name, "prediction_path": pred_path, **report_meta}


## Train

In [9]:
# Configureations and select models to train
AF_SELECTED_MODELS = [
    "Wav2Vec2-base-attnstat",
    "WavLM-base-attnstat",
    "Whisper-base",
    "HuBERT-base-attnstat",
]

# All 10 folds
AF_SELECTED_FOLDS = None
AF_SMOKE_RUN = False
AF_BUILD_ENSEMBLES_AFTER_RUN = True

In [10]:
af_exp_by_name = {cfg.short_name: cfg for cfg in AF_EXPERIMENTS}
af_completed_summaries = []

for _name in AF_SELECTED_MODELS:
    if _name not in af_exp_by_name:
        print(f"[SKIP] Unknown AF experiment name: {_name}")
        continue
    _cfg = af_exp_by_name[_name]
    try:
        _summary = af_run_experiment(
            _cfg,
            fold_indices=AF_SELECTED_FOLDS,
            smoke=AF_SMOKE_RUN,
        )
        af_completed_summaries.append(_summary)
        af_write_summary_csv(af_completed_summaries)
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            print(f"[OOM] {_cfg.short_name}: {exc}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        else:
            raise
    except Exception as exc:
        print(f"[FAILED] {_cfg.short_name}: {type(exc).__name__}: {exc}")

print(f"Completed pretrained runs in this notebook session: {len(af_completed_summaries)}")



  Wav2Vec2-base-attnstat | Fold 01 | test=03 val=08


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 42919.41it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch 01 | train_loss=1.9321 train_acc=0.196 | val_loss=1.8340 val_acc=0.190 val_macro_f1=0.156 | wait=0
  Epoch 02 | train_loss=1.6385 train_acc=0.395 | val_loss=1.4733 val_acc=0.483 val_macro_f1=0.421 | wait=0
  Epoch 03 | train_loss=1.1718 train_acc=0.607 | val_loss=1.2208 val_acc=0.603 val_macro_f1=0.519 | wait=0
  Epoch 04 | train_loss=0.9014 train_acc=0.706 | val_loss=1.1049 val_acc=0.724 val_macro_f1=0.659 | wait=0
  Epoch 05 | train_loss=0.6981 train_acc=0.832 | val_loss=0.9986 val_acc=0.793 val_macro_f1=0.727 | wait=0
  Epoch 06 | train_loss=0.6426 train_acc=0.839 | val_loss=0.9703 val_acc=0.741 val_macro_f1=0.666 | wait=1
  Epoch 07 | train_loss=0.5564 train_acc=0.907 | val_loss=0.7593 val_acc=0.776 val_macro_f1=0.705 | wait=2
  Epoch 08 | train_loss=0.5193 train_acc=0.895 | val_loss=0.6810 val_acc=0.862 val_macro_f1=0.768 | wait=0
  Epoch 09 | train_loss=0.4542 train_acc=0.937 | val_loss=0.7198 val_acc=0.862 val_macro_f1=0.771 | wait=0
  Epoch 10 | train_loss=0.4157 train_

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 65049.48it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch 01 | train_loss=1.9523 train_acc=0.143 | val_loss=1.7744 val_acc=0.326 val_macro_f1=0.254 | wait=0
  Epoch 02 | train_loss=1.6857 train_acc=0.364 | val_loss=1.3575 val_acc=0.465 val_macro_f1=0.378 | wait=0
  Epoch 03 | train_loss=1.2046 train_acc=0.592 | val_loss=0.9826 val_acc=0.791 val_macro_f1=0.679 | wait=0
  Epoch 04 | train_loss=0.8943 train_acc=0.726 | val_loss=0.7479 val_acc=0.860 val_macro_f1=0.737 | wait=0
  Epoch 05 | train_loss=0.7051 train_acc=0.825 | val_loss=0.6382 val_acc=0.814 val_macro_f1=0.766 | wait=0
  Epoch 06 | train_loss=0.6324 train_acc=0.873 | val_loss=0.6913 val_acc=0.837 val_macro_f1=0.791 | wait=0
  Epoch 07 | train_loss=0.5344 train_acc=0.906 | val_loss=0.5272 val_acc=0.884 val_macro_f1=0.827 | wait=0
  Epoch 08 | train_loss=0.4676 train_acc=0.952 | val_loss=0.4898 val_acc=0.930 val_macro_f1=0.900 | wait=0
  Epoch 09 | train_loss=0.4363 train_acc=0.942 | val_loss=0.5657 val_acc=0.884 val_macro_f1=0.765 | wait=1
  Epoch 10 | train_loss=0.3942 train_

...

  Epoch 01 | train_loss=1.9706 train_acc=0.171 | val_loss=1.8947 val_acc=0.429 val_macro_f1=0.238 | wait=0
  Epoch 02 | train_loss=1.7399 train_acc=0.346 | val_loss=1.4484 val_acc=0.518 val_macro_f1=0.363 | wait=0
  Epoch 03 | train_loss=1.3184 train_acc=0.522 | val_loss=1.1665 val_acc=0.536 val_macro_f1=0.442 | wait=0
  Epoch 04 | train_loss=1.0397 train_acc=0.649 | val_loss=1.2510 val_acc=0.536 val_macro_f1=0.451 | wait=0
  Epoch 05 | train_loss=0.9176 train_acc=0.678 | val_loss=1.2780 val_acc=0.589 val_macro_f1=0.533 | wait=0
  Epoch 06 | train_loss=0.7699 train_acc=0.788 | val_loss=0.8147 val_acc=0.786 val_macro_f1=0.762 | wait=0
  Epoch 07 | train_loss=0.6875 train_acc=0.817 | val_loss=0.9198 val_acc=0.839 val_macro_f1=0.823 | wait=0
  Epoch 08 | train_loss=0.5862 train_acc=0.885 | val_loss=0.7686 val_acc=0.804 val_macro_f1=0.746 | wait=1
  Epoch 09 | train_loss=0.5563 train_acc=0.898 | val_loss=0.9835 val_acc=0.804 val_macro_f1=0.766 | wait=2
  Epoch 10 | train_loss=0.5052 train_

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 90315.15it/s]


  Epoch 01 | train_loss=1.9808 train_acc=0.132 | val_loss=1.8533 val_acc=0.408 val_macro_f1=0.271 | wait=0
  Epoch 02 | train_loss=1.8033 train_acc=0.321 | val_loss=1.6522 val_acc=0.380 val_macro_f1=0.277 | wait=0
  Epoch 03 | train_loss=1.4116 train_acc=0.485 | val_loss=1.4084 val_acc=0.493 val_macro_f1=0.359 | wait=0
  Epoch 04 | train_loss=1.0914 train_acc=0.637 | val_loss=1.1988 val_acc=0.620 val_macro_f1=0.523 | wait=0
  Epoch 05 | train_loss=0.9509 train_acc=0.659 | val_loss=1.1143 val_acc=0.648 val_macro_f1=0.526 | wait=0
  Epoch 06 | train_loss=0.8139 train_acc=0.770 | val_loss=0.9873 val_acc=0.732 val_macro_f1=0.617 | wait=0
  Epoch 07 | train_loss=0.7519 train_acc=0.777 | val_loss=0.9093 val_acc=0.718 val_macro_f1=0.683 | wait=0
  Epoch 08 | train_loss=0.6314 train_acc=0.870 | val_loss=0.9699 val_acc=0.718 val_macro_f1=0.683 | wait=1
  Epoch 09 | train_loss=0.5771 train_acc=0.880 | val_loss=0.9237 val_acc=0.718 val_macro_f1=0.671 | wait=2
  Epoch 10 | train_loss=0.5466 train_

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 71061.36it/s]


  Epoch 01 | train_loss=1.9778 train_acc=0.154 | val_loss=1.9238 val_acc=0.143 val_macro_f1=0.036 | wait=0
  Epoch 02 | train_loss=1.8367 train_acc=0.296 | val_loss=1.5117 val_acc=0.469 val_macro_f1=0.244 | wait=0
  Epoch 03 | train_loss=1.4713 train_acc=0.484 | val_loss=1.2472 val_acc=0.551 val_macro_f1=0.405 | wait=0
  Epoch 04 | train_loss=1.0949 train_acc=0.629 | val_loss=1.0959 val_acc=0.633 val_macro_f1=0.542 | wait=0
  Epoch 05 | train_loss=0.9214 train_acc=0.716 | val_loss=1.0715 val_acc=0.694 val_macro_f1=0.583 | wait=0
  Epoch 06 | train_loss=0.7950 train_acc=0.752 | val_loss=1.1249 val_acc=0.714 val_macro_f1=0.648 | wait=0
  Epoch 07 | train_loss=0.6618 train_acc=0.841 | val_loss=0.9147 val_acc=0.694 val_macro_f1=0.616 | wait=1
  Epoch 08 | train_loss=0.6080 train_acc=0.863 | val_loss=1.0139 val_acc=0.796 val_macro_f1=0.742 | wait=0
  Epoch 09 | train_loss=0.5740 train_acc=0.882 | val_loss=1.1685 val_acc=0.735 val_macro_f1=0.662 | wait=1
  Epoch 10 | train_loss=0.5385 train_

## Evaluation (Single model + Ensemble)

In [11]:
if AF_BUILD_ENSEMBLES_AFTER_RUN:
    _full_runs = sorted(
        [
            s for s in af_completed_summaries
            if s.get("num_folds") == NUM_FOLDS and not AF_SMOKE_RUN
        ],
        key=lambda s: (s.get("macro_f1", 0), s.get("accuracy", 0)),
        reverse=True,
    )

    af_ensemble_summaries = []
    for _k in (2, 3, len(_full_runs)):
        if len(_full_runs) >= _k >= 2:
            _files = [s["prediction_path"] for s in _full_runs[:_k]]
            af_ensemble_summaries.append(
                af_ensemble_from_prediction_files(_files, f"Pretrained-Ensemble-top{_k}")
            )

    if af_ensemble_summaries:
        af_write_summary_csv(af_completed_summaries + af_ensemble_summaries)


_summary_path = os.path.join(AF_RESULTS_DIR, "pretrained_loso_summary.csv")

with open(_summary_path, newline="", encoding="utf-8") as _f:
    _rows = list(csv.DictReader(_f))

print(f"{'Model':<38} {'Accuracy':>10} {'Macro-F1':>10} {'Folds':>7}")
print("-" * 72)

for _row in sorted(_rows, key=lambda r: float(r.get("accuracy") or 0), reverse=True):
    _acc = float(_row.get("accuracy") or 0) * 100
    _f1 = float(_row.get("macro_f1") or 0)
    print(
        f"{_row.get('short_name',''):<38} "
        f"{_acc:>9.2f}% "
        f"{_f1:>10.4f} "
        f"{_row.get('num_folds',''):>7}"
    )


Summary CSV saved: results\pretrained_loso_summary.csv
Model                                    Accuracy   Macro-F1   Folds
------------------------------------------------------------------------
Pretrained-Ensemble-top4                   90.65%     0.9036        
Pretrained-Ensemble-top2                   89.91%     0.8971        
Pretrained-Ensemble-top3                   89.72%     0.8962        
WavLM-base-attnstat                        86.54%     0.8633      10
Wav2Vec2-base-attnstat                     85.98%     0.8558      10
HuBERT-base-attnstat                       85.79%     0.8544      10
Whisper-base                               82.99%     0.8219      10
